<a href="https://colab.research.google.com/github/FranciscoBPereira/AnaliseDados_MEI_2526/blob/main/AD2526_P4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**A. CIFAR 100 Dataset: Fine grained Version**

**0. Obtaining and Splitting the Dataset into 3 Sets (Train, Test, Validation)**


In [ ]:

# Load CIFAR100 dataset from keras datasets:
# https://keras.io/api/datasets/cifar100/
# https://www.cs.toronto.edu/~kriz/cifar.html

# The load_data() method creates train and test sets. The parameter label_mode specifies the category labels: 'fine' or 'coarse'
# In this class we will adopt the fine grained classification, corresponding to 100 categories

from keras.datasets import cifar100
from sklearn.model_selection import train_test_split

(train_images_full, train_labels_full), (test_images, test_labels) = cifar100.load_data(label_mode = 'fine')

train_labels_full = train_labels_full.squeeze()
test_labels = test_labels.squeeze()


# We further divide the original train datasets into train and validation datasets
train_images, valid_images, train_labels, valid_labels = train_test_split(
    train_images_full, train_labels_full,
    test_size=0.1,
    random_state=42,
    stratify=train_labels_full
)

# Normalize data to interval [0, 1]

train_images = train_images / 255.0
valid_images = valid_images / 255.0
test_images = test_images / 255.0


**1. Baseline CNN**

In [ ]:
# Last class we struggled to achieve 30% in test accuracy when using a MLP
# Let's check how a plain CNN performs. No regularization or overfitting control techniques will be used.

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

CNN_Base = keras.Sequential([
    # Input layer (No Flatten layer)
    layers.Input(shape=[32,32,3]),

    # Feature extraction part
    layers.Conv2D(32, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),

    #Classification part
    layers.Flatten(),
    layers.Dense(64, activation='elu'),

    # Final layer
    layers.Dense(100, activation='softmax')
])


CNN_Base.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])

CNN_Base.summary()

history = CNN_Base.fit(train_images, train_labels, batch_size=32, epochs=30,
                    validation_data=(valid_images, valid_labels))




In [ ]:
x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()

test_loss, test_acc = CNN_Base.evaluate(test_images, test_labels)
print(f"Test Accuracy: {test_acc}")

**B. Flowers Dataset**

**1. Data Fetching and Loading**

In [ ]:
# Download the Flowers dataset: https://www.kaggle.com/datasets/imsparsh/flowers-dataset

# Create folder flower_photos
# Inside this folder there are 5 subfolders, one for each category

!curl -O https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz

!tar -xzvf flower_photos.tgz

!rm flower_photos.tgz


In [ ]:
# Check the total number of images

import pathlib

os.chdir('flower_photos')
path = os.getcwd()
data_dir = pathlib.Path(path)

image_count = len(list(data_dir.glob('*/*.jpg')))
print('Total images: ', image_count)

In [ ]:
# The images in the original folders are not divided in train and validation datasets
# The following code divides samples into 80% training and 20% validation. No test set is created

# https://www.tensorflow.org/api_docs/python/tf/data/Dataset
# https://www.tensorflow.org/guide/data

# The method image_dataset_from_directory() creates Dataset objects from images located in a specified directory
# https://keras.io/api/data_loading/image/

# Parameters subset and validation_split enable the creation of two datasets (Train: 80%; Validation: 20%)
# This method may shuffle images, adjust size and define the batch size
# This way the dataset is (almost) ready to be processed by the neural network


batch_size = 32
IMG_SIZE = (180, 180)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=batch_size,
)

class_names = train_ds.class_names


train_ds = train_ds.cache().prefetch(1)
val_ds = val_ds.cache().prefetch(1)

In [ ]:
# Dataset detailed information

print('Nr. of classes: ', len(class_names))
print('Classes: ', class_names, '\n')

# Cardinality
print('Cardinalidade Treino: ', train_ds.cardinality().numpy())
print('Cardinalidade Validacão: ', val_ds.cardinality().numpy(), '\n')

for image_batch, labels_batch in train_ds:
    print('Tensor Batch Image: ', image_batch.shape)
    print('Tensor Batch Label: ', labels_batch.shape)
    break

In [ ]:
# Visualize a few examples

plt.figure(figsize=(12, 12))
for images, labels in train_ds.take(1):
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy()/255.)
        plt.title(class_names[labels[i]])
        plt.axis("off")

**2. Creating and Training a Baseline CNN: CNN_Flower1**

In [ ]:
# Adapt the baseline CNN that we used in the CIFAR100 dataset to this new example
# For now, change just the input and output layers
# Note that the rescaling is performed by the neural network

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

CNN_Flower1 = keras.Sequential([

    ### Add Input layer ####

    layers.Rescaling(scale = 1./255),


    # Feature extraction part

    layers.Conv2D(32, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(32, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='elu', padding='same'),
    layers.MaxPooling2D(),


    #Classification part
    layers.Flatten(),
    layers.Dense(64, activation='elu'),

    ### Add Final layer ###
])


CNN_Flower1.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])

CNN_Flower1.summary()

history = CNN_Flower1.fit(
  train_ds,
  validation_data=val_ds,
  epochs=30
)


In [ ]:

x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()

**Question:**

How do you analyze results?

**3. Creating and Training an Improved CNN: CNN_Flower2**

In [ ]:
# Design and implement changes in the CNN and repeat the training process,
# seeking for an architecture that performs more effectively.

# Among other possibilities, you might consider one of the following points:
#  1. Change the CNN architecture, adding, deleting, or changing the parameterization of convolutional, maxpooling or dense layers.
#  2. Change the Optimizer
#  3. Add Regularization, Batch Normalization or Dropout layers.
#  4. Add a callback to implement Early Stopping.

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)


# Create a new model called CNN_Flower2





CNN_Flower2 = keras.Sequential([

     ###  CODE GOES HERE   ####

])



CNN_Flower2.compile(loss='sparse_categorical_crossentropy',
              optimizer=keras.optimizers.SGD(),
              metrics=['accuracy'])

CNN_Flower2.summary()

history = CNN_Flower2.fit(
  train_ds,
  validation_data=val_ds,
  epochs=30
)



In [ ]:
x = pd.DataFrame(history.history, columns = ['accuracy', 'val_accuracy'])
x.plot(figsize=(8, 5))
plt.grid(True)
plt.show()